In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

# -------------------------------------------------
# INITIAL SETUP
# -------------------------------------------------
fake = Faker("en_IN")
np.random.seed(42)
random.seed(42)

NUM_ROWS = 250000
CURRENT_DATE = datetime(2025, 1, 1)

# -------------------------------------------------
# MUMBAI PINCODES
# -------------------------------------------------
mumbai_pincodes = [
    400001, 400002, 400004, 400008, 400012,
    400016, 400020, 400022, 400026, 400050,
    400053, 400056, 400076, 400080
]

# -------------------------------------------------
# BLOOD GROUP DISTRIBUTION (India Approx %)
# -------------------------------------------------
blood_distribution = {
    "O+": 37,
    "B+": 32,
    "A+": 22,
    "AB+": 7,
    "O-": 1,
    "A-": 0.5,
    "B-": 0.5,
    "AB-": 0.1
}

blood_types = list(blood_distribution.keys())
blood_probs = np.array(list(blood_distribution.values()))
blood_probs = blood_probs / blood_probs.sum()  # Normalize properly

# -------------------------------------------------
# DATA GENERATION
# -------------------------------------------------
rows = []

for i in range(NUM_ROWS):

    # -------------------------
    # AGE (18–75)
    # -------------------------
    age = np.random.randint(18, 76)

    # -------------------------
    # TOTAL DONATIONS correlated with age
    # -------------------------
    if age < 25:
        total_donations = np.random.poisson(1)
    elif age < 35:
        total_donations = np.random.poisson(3)
    elif age < 50:
        total_donations = np.random.poisson(6)
    else:
        total_donations = np.random.poisson(9)

    total_donations = max(0, total_donations)

    # -------------------------
    # LAST DONATION DATE (within last 900 days)
    # -------------------------
    days_ago = np.random.randint(0, 900)
    last_donation = CURRENT_DATE - timedelta(days=int(days_ago))

    # -------------------------
    # DAYS SINCE DONATION
    # -------------------------
    days_since = (CURRENT_DATE - last_donation).days

    # -------------------------
    # DONOR TIER LOGIC
    # -------------------------
    if total_donations <= 1:
        tier = "New"
    elif total_donations <= 4:
        tier = "Silver"
    elif total_donations <= 9:
        tier = "Gold"
    else:
        tier = "Platinum"

    # -------------------------
    # ELIGIBILITY RULE (90 days gap)
    # -------------------------
    eligible = 1 if days_since >= 90 else 0

    # -------------------------
    # AVAILABILITY PROBABILITY (Bell logic)
    # -------------------------
    if days_since < 90:
        availability = np.random.uniform(0.05, 0.25)
    elif days_since < 300:
        availability = np.random.uniform(0.6, 0.9)
    else:
        availability = np.random.uniform(0.2, 0.5)

    # Boost high tier donors
    if tier in ["Gold", "Platinum"]:
        availability += 0.05

    availability = round(min(1.0, availability), 2)

    # -------------------------
    # OPTIONAL PREDICTIVE TARGET
    # Will donate next 30 days (Binary)
    # -------------------------
    will_donate_next_30_days = 1 if random.random() < availability else 0

    # -------------------------
    # APPEND ROW
    # -------------------------
    rows.append([
        f"MUM-D-{100000+i}",
        fake.name(),
        age,
        random.choice(["Male", "Female"]),
        np.random.choice(blood_types, p=blood_probs),
        random.choice(mumbai_pincodes),
        last_donation.strftime("%d-%m-%Y"),
        total_donations,
        days_since,
        tier,
        eligible,
        availability,
        will_donate_next_30_days
    ])

# -------------------------------------------------
# CREATE DATAFRAME
# -------------------------------------------------
columns = [
    "Donor_ID",
    "Name",
    "Age",
    "Gender",
    "Blood_Type",
    "Location_Pincode",
    "Last_Donation_Date",
    "Total_Donations",
    "Days_Since_Donation",
    "Donor_Tier",
    "Eligible",
    "Availability_Probability",
    "Will_Donate_Next_30_Days"
]

df = pd.DataFrame(rows, columns=columns)

# -------------------------------------------------
# SAVE DATASET
# -------------------------------------------------
df.to_csv("synthetic_mumbai_donors_250k.csv", index=False)

print("Dataset successfully generated!")
print("Total Rows:", len(df))

Dataset successfully generated!
Total Rows: 250000
